# Concept drift check

Detailed workflow for this experiment. For catalogue inputs and dependencies, see [data](../../../docs/data.ipynb). Enable `LG_RUN_RESEARCH_WORKFLOW=1` before running the calculation cells.

In [ ]:
import os
RUN_RESEARCH_WORKFLOW = os.environ.get("LG_RUN_RESEARCH_WORKFLOW") == "1"
if not RUN_RESEARCH_WORKFLOW:
    raise RuntimeError("Optional workflow disabled. Read notebooks/92_reproduce_saved_analyses.ipynb and set LG_RUN_RESEARCH_WORKFLOW=1.")

In [ ]:
from pathlib import Path
import os
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").is_file())
os.environ.setdefault("LG_PROJECT", str(ROOT / "research"))
PROJECT = Path(os.environ["LG_PROJECT"]).expanduser().resolve()
os.chdir(PROJECT)

def design_path(logical_name):
    if logical_name == "FINAL_EXPERIMENT_DESIGN_2026-10-01.md":
        return PROJECT / "out/final_stage_audit/portability_v3/original_design.txt"
    return PROJECT / logical_name

# Corrected TNG300 comparisons

Restart the kernel and run from the top. Set `LG_PROJECT` to your project directory on Roihu before running the setup cell. Run the v3 inference notebook first to create the checkpoints; use the original Roihu environment.

Inputs are verified against the v3 manifests. The corrected TNG sample has 1,530 pairs: 1,253 same-host and 277 different-host. Figures are written to `pickles_v3/figures/Concept_drift_check`. Selection and feature caches are read only. Both simulations' radial pair velocities are **peculiar**; these comparison plots do not add a Hubble term.

Pair distributions count one row per pair. Galaxy-feature distributions explicitly distinguish unique final-pair members from pair-member entries (a galaxy may appear repeatedly). KS/energy/Wasserstein distances are descriptive; shared galaxies and hosts invalidate naive independent-row significance claims. Historical interpretation notes below require reassessment after rerunning.

Raw SOAP metadata, raw TNG catalogue checks, and large parent-catalogue diagnostics are optional flags in the setup cell and are off by default. Existing originals and outputs are preserved in the dated comparison-update backup.


In [ ]:
import os, sys
from pathlib import Path
PROJECT = Path(os.environ.get("LG_PROJECT", r"<research-project>"))
sys.path.insert(0, str(PROJECT / "analogues"))
from review_fixes import (RUN_VERSION, MODEL_DESCRIPTION, H0_BY_SIM, configure_run,
    feature_cache_dir, lg_observation, lg_to_peculiar, draw_noisy_feature,
    sample_lg_point, sample_lg_marginalized, calibration_summary, tng_volume_mpc3)
LEGACY_RESULTS_DIR = PROJECT / "pickles_v2"
FEATURE_CACHE_DIR = feature_cache_dir(PROJECT)

from review_fixes import require_legacy_environment
require_legacy_environment(PROJECT)

from comparison_utils import (load_comparison_inputs, finite_lookup,
    normalized_wasserstein, group_blocks, resample_blocks)
from splits import pair_groups, group_split
RESULTS_DIR, PLOT_DIR, tng_dir, TNG_DATA, TNG_MANIFEST = load_comparison_inputs(
    PROJECT, 'Concept_drift_check')
RUN_SOAP_METADATA = False  # Optional remote metadata lookup; requires hdfstream.
RUN_TNG_CATALOGUE_CHECKS = False  # Optional raw catalogue; set TNG_BASE when enabled.
RUN_PARENT_CATALOGUE_CHECKS = False  # Optional large FLAMINGO parent-cache diagnostic.

In [ ]:
import os, sys
from pathlib import Path
PROJECT  = Path(os.environ.get('LG_PROJECT', r'<research-project>'))
TNG_BASE = os.environ.get('TNG_BASE', r'<TNG_BASE>')
sys.path.insert(0, str(PROJECT / 'analogues'))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import functools
from pathlib import Path
import torch
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
import sys
sys.path.insert(0, str(PROJECT / "analogues"))
from splits import pair_groups, group_split

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
results_dir = RESULTS_DIR
results_dir.mkdir(parents=True, exist_ok=True)
cache_dir   = PROJECT / "analogues/cache/run_001_mstar2e10"
tng_dir     = PROJECT / "analogues/tng_for_comparison_v3"

def load_checkpoint_any_device(path, device):
    original_torch_load = torch.load
    torch.load = functools.partial(original_torch_load, map_location=device)
    try:
        with open(path, "rb") as f:
            obj = pickle.load(f)
    finally:
        torch.load = original_torch_load
    return obj

flam_ckpt = load_checkpoint_any_device(results_dir / "flamingo_npe_checkpoint.pkl", device)
flam_posteriors = flam_ckpt["posteriors"]
flam_scalers    = flam_ckpt["scalers"]

tng_ckpt = load_checkpoint_any_device(results_dir / "tng_npe_checkpoint.pkl", device)
tng_posteriors = tng_ckpt["posteriors"]
tng_scalers    = tng_ckpt["scalers"]

BLOCKS = {
    "core_dynamics":   ["separation", "vr", "vt", "log_mass_ratio"],
    "core_chemistry":  ["separation", "vr", "vt", "log_mass_ratio", "met_diff"],
    "core_morphology": ["separation", "vr", "vt", "log_mass_ratio",
                        "sigma_mean", "sigma_diff", "kappa_mean", "kappa_diff"],
    "core_photometry": ["separation", "vr", "vt", "log_mass_ratio", "gr_mean", "gr_diff"],
    "all_features":    ["separation", "vr", "vt", "log_mass_ratio",
                        "sigma_mean", "sigma_diff", "kappa_mean", "kappa_diff",
                        "met_diff", "gr_mean", "gr_diff"],
}
TARGET = "log_m200c"

# raw data
with open(cache_dir / "flamingo_sample.pkl", "rb") as f:
    sample = pickle.load(f)
with open(cache_dir / "flamingo_pairs.pkl", "rb") as f:
    pairs = pickle.load(f)
with open(FEATURE_CACHE_DIR / "metallicity_log_zscore.pkl", "rb") as f:
    met_transform = pickle.load(f)
with open(FEATURE_CACHE_DIR / "tng_sigma_star.pkl", "rb") as f:
    tng_sigma_star = pickle.load(f)
with open(FEATURE_CACHE_DIR / "tng_kappa_corot.pkl", "rb") as f:
    kappa_cache = pickle.load(f)

def pair_features_flamingo(sample, pairs, met_transform):
    pi, pj = pairs["i"], pairs["j"]
    gr_i, gr_j = sample["g_r"][pi], sample["g_r"][pj]
    sigma_i, sigma_j = sample["vel_disp"][pi], sample["vel_disp"][pj]
    kap_i, kap_j = sample["kappa_rot"][pi], sample["kappa_rot"][pj]
    met_i = met_transform["flamingo_met_dex"][pi]
    met_j = met_transform["flamingo_met_dex"][pj]
    return pd.DataFrame({
        "gr_mean": (gr_i+gr_j)/2, "gr_diff": np.abs(gr_i-gr_j),
        "met_mean": (met_i+met_j)/2, "met_diff": np.abs(met_i-met_j),
        "sigma_mean": (sigma_i+sigma_j)/2, "sigma_diff": np.abs(sigma_i-sigma_j),
        "kappa_mean": (kap_i+kap_j)/2, "kappa_diff": np.abs(kap_i-kap_j),
        "separation": pairs["separation_kpc"], "vr": pairs["vr"], "vt": pairs["vt"],
        "log_mass_ratio": pairs["log_mass_ratio"], "log_m200c": pairs["log_tot_virial_mass"],
        "host_i": sample["host_top_index"][pi],
        "host_j": sample["host_top_index"][pj],
    })

flam_df = pair_features_flamingo(sample, pairs, met_transform)
flam_df["same_host"] = pairs["have_same_host"].astype(bool)

keep_idx    = TNG_DATA["keep_idx"]
tng_pairs_i = TNG_DATA["pairs_i"]
tng_pairs_j = TNG_DATA["pairs_j"]
tng_gr_sample   = TNG_DATA["g_r"]
tng_met_sample  = TNG_DATA["metallicity"]
tng_separation  = TNG_DATA["separation"]
tng_vr          = TNG_DATA["vr"]
tng_vt          = TNG_DATA["vt"]
tng_log_m200c   = TNG_DATA["log_m200c"]
tng_log_mass_ratio = TNG_DATA["log_mass_ratio"]
tng_grnr = TNG_DATA["grnr"]

tng_pairs_i_global = keep_idx[tng_pairs_i]
tng_pairs_j_global = keep_idx[tng_pairs_j]
tng_sigma_i = finite_lookup(tng_sigma_star, tng_pairs_i_global, "tng_sigma_star")
tng_sigma_j = finite_lookup(tng_sigma_star, tng_pairs_j_global, "tng_sigma_star")
tng_kappa_i = finite_lookup(kappa_cache, tng_pairs_i_global, "kappa_cache")
tng_kappa_j = finite_lookup(kappa_cache, tng_pairs_j_global, "kappa_cache")

def pair_features_tng(tng_pairs_i, tng_pairs_j, tng_gr_sample, tng_met_sample,
                       tng_sigma_i, tng_sigma_j, tng_kappa_i, tng_kappa_j,
                       tng_separation, tng_vr, tng_vt, tng_log_mass_ratio,
                       tng_log_m200c, met_transform):
    gr_i, gr_j = tng_gr_sample[tng_pairs_i], tng_gr_sample[tng_pairs_j]
    met_raw_i, met_raw_j = tng_met_sample[tng_pairs_i], tng_met_sample[tng_pairs_j]
    met_i = met_transform["tng_met_dex"][tng_pairs_i]
    met_j = met_transform["tng_met_dex"][tng_pairs_j]
    return pd.DataFrame({
        "gr_mean": (gr_i+gr_j)/2, "gr_diff": np.abs(gr_i-gr_j),
        "met_mean": (met_i+met_j)/2, "met_diff": np.abs(met_i-met_j),
        "sigma_mean": (tng_sigma_i+tng_sigma_j)/2, "sigma_diff": np.abs(tng_sigma_i-tng_sigma_j),
        "kappa_mean": (tng_kappa_i+tng_kappa_j)/2, "kappa_diff": np.abs(tng_kappa_i-tng_kappa_j),
        "separation": tng_separation, "vr": tng_vr, "vt": tng_vt,
        "log_mass_ratio": tng_log_mass_ratio, "log_m200c": tng_log_m200c,
        "host_i": tng_grnr[tng_pairs_i],
        "host_j": tng_grnr[tng_pairs_j],
    })

tng_df = pair_features_tng(
    tng_pairs_i, tng_pairs_j, tng_gr_sample, tng_met_sample,
    tng_sigma_i, tng_sigma_j, tng_kappa_i, tng_kappa_j,
    tng_separation, tng_vr, tng_vt, tng_log_mass_ratio,
    tng_log_m200c, met_transform
)

ALL_FEATURES = BLOCKS["all_features"]
flam_df_clean = flam_df.dropna(subset=ALL_FEATURES + [TARGET])
tng_df_clean  = tng_df.dropna(subset=ALL_FEATURES + [TARGET])
flam_diff = flam_df_clean[~flam_df_clean["same_host"]].copy()

def get_test_split(df, feats, seed=42, test_size=0.15):
    cols  = feats + [TARGET, "host_i", "host_j"]
    valid = df[cols].dropna(subset=feats + [TARGET]).reset_index(drop=True)
    _, te = group_split(valid, test_size=test_size, seed=seed)
    return valid.iloc[te]

DATASETS = {"full": flam_df_clean, "diff_host": flam_diff}
print(f"FLAMINGO: {len(flam_df_clean):,}   TNG: {len(tng_df_clean):,}")
assert len(tng_df_clean) == TNG_MANIFEST["pair_count"], "Unexpected loss of corrected TNG rows."

In [ ]:
# Intervals condition on the fitted model and resample connected-host components.
def posterior_point_predictions(posterior, X_raw, scaler, device, n_samples=500):
    X_sc = scaler.transform(X_raw.astype(np.float32))
    x_batch = torch.tensor(X_sc, device=device)
    with torch.no_grad():
        samples = posterior.sample_batched((n_samples,), x=x_batch)
    return samples.mean(dim=0).squeeze(-1).cpu().numpy()

def bootstrap_r2_ci(posterior, scaler, X_raw, y_true, groups, n_boot=1000, seed=0):
    rng = np.random.default_rng(seed)
    preds = posterior_point_predictions(posterior, X_raw, scaler, device)
    blocks = group_blocks(groups)
    if len(groups) != len(y_true):
        raise ValueError("Host groups and targets are not aligned.")
    boot_r2 = np.empty(n_boot)
    for b in range(n_boot):
        idx = resample_blocks(blocks, rng)
        boot_r2[b] = r2_score(y_true[idx], preds[idx]) if np.var(y_true[idx]) > 0 else np.nan
    boot_r2 = boot_r2[np.isfinite(boot_r2)]
    if not len(boot_r2):
        raise ValueError("No nondegenerate host-bootstrap draws.")
    point_r2 = r2_score(y_true, preds)
    ci = (np.percentile(boot_r2, 2.5), np.percentile(boot_r2, 97.5))
    return point_r2, ci

print(f"{'Block':<20} {'Held-out R² [95% CI]':>30} {'Cross-sim R² [95% CI]':>30}")

r2_ci_results = {}
for block_name, feats in BLOCKS.items():
    key = f"{block_name}__full"
    posterior, scaler = flam_posteriors[key], flam_scalers[key]

    test_flam = get_test_split(flam_df_clean, feats)
    held_r2, held_ci = bootstrap_r2_ci(posterior, scaler, test_flam[feats].values, test_flam[TARGET].values, pair_groups(test_flam))

    tng_valid = tng_df_clean[feats + [TARGET, "host_i", "host_j"]].dropna()
    cross_r2, cross_ci = bootstrap_r2_ci(posterior, scaler, tng_valid[feats].values, tng_valid[TARGET].values, pair_groups(tng_valid))

    r2_ci_results[block_name] = {"held_out": (held_r2, held_ci), "cross_sim": (cross_r2, cross_ci)}
    print(f"{block_name:<20} {held_r2:>8.3f} [{held_ci[0]:.3f}, {held_ci[1]:.3f}]      "
          f"{cross_r2:>8.3f} [{cross_ci[0]:.3f}, {cross_ci[1]:.3f}]")

with open(results_dir / "r2_bootstrap_ci.pkl", "wb") as f:
    pickle.dump(r2_ci_results, f)

In [ ]:
def slope_comparison(flam_df, tng_df, feature, target=TARGET):
    flam_valid = flam_df[[feature, target]].dropna()
    tng_valid  = tng_df[[feature, target]].dropna()

    flam_X = flam_valid[[feature]].values
    flam_y = flam_valid[target].values
    tng_X  = tng_valid[[feature]].values
    tng_y  = tng_valid[target].values

    flam_fit = LinearRegression().fit(flam_X, flam_y)
    tng_fit  = LinearRegression().fit(tng_X, tng_y)

    return {
        "flam_slope": flam_fit.coef_[0], "flam_intercept": flam_fit.intercept_,
        "tng_slope": tng_fit.coef_[0], "tng_intercept": tng_fit.intercept_,
        "slope_diff": flam_fit.coef_[0] - tng_fit.coef_[0],
    }

def bootstrap_slope_diff_ci(flam_df, tng_df, feature, target=TARGET, n_boot=1000, seed=0):
    rng = np.random.default_rng(seed)
    flam_rows = flam_df[[feature, target, "host_i", "host_j"]].dropna()
    tng_rows = tng_df[[feature, target, "host_i", "host_j"]].dropna()
    flam_blocks, tng_blocks = group_blocks(pair_groups(flam_rows)), group_blocks(pair_groups(tng_rows))
    flam_valid = flam_rows[[feature, target]].values
    tng_valid = tng_rows[[feature, target]].values
    diffs = np.empty(n_boot)
    for b in range(n_boot):
        f_idx = resample_blocks(flam_blocks, rng)
        t_idx = resample_blocks(tng_blocks, rng)
        f_fit = LinearRegression().fit(flam_valid[f_idx][:, [0]], flam_valid[f_idx, 1])
        t_fit = LinearRegression().fit(tng_valid[t_idx][:, [0]], tng_valid[t_idx, 1])
        diffs[b] = f_fit.coef_[0] - t_fit.coef_[0]
    return diffs

for feat in ["sigma_mean", "kappa_mean"]:
    result = slope_comparison(flam_df_clean, tng_df_clean, feat)
    diffs = bootstrap_slope_diff_ci(flam_df_clean, tng_df_clean, feat)
    ci = (np.percentile(diffs, 2.5), np.percentile(diffs, 97.5))
    crosses_zero = ci[0] <= 0 <= ci[1]

    print(f"\n{feat}:")
    print(f"  FLAMINGO slope: {result['flam_slope']:.5f}  (log_m200c per unit {feat})")
    print(f"  TNG slope:      {result['tng_slope']:.5f}")
    print(f"  Difference:     {result['slope_diff']:.5f}   95% CI: [{ci[0]:.5f}, {ci[1]:.5f}]")
    print(f"  {'Marginal slope-contrast CI includes zero' if crosses_zero else 'Marginal slope-contrast CI excludes zero; this alone does not establish conditional concept drift'}")

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, feat in zip(axes, ["sigma_mean", "kappa_mean"]):
    for df, color, label in [(flam_df_clean, "tab:orange", "FLAMINGO"), (tng_df_clean, "tab:blue", "TNG")]:
        valid = df[[feat, TARGET]].dropna()
        ax.scatter(valid[feat], valid[TARGET], s=2, alpha=0.15, color=color)
        fit = LinearRegression().fit(valid[[feat]].values, valid[TARGET].values)
        xs_line = np.linspace(valid[feat].min(), valid[feat].max(), 100)
        ax.plot(xs_line, fit.predict(xs_line.reshape(-1,1)), color=color, lw=2, label=label)
    ax.set_xlabel(feat); ax.set_ylabel("log_m200c"); ax.set_title(feat); ax.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / "comparison_06.pdf", bbox_inches="tight")
plt.show()